# 12 — Evaluate behavior, not just prose
Objective checks should be plain Python. A model-judged score can be added
later, but approval, citation IDs, and bounded retries must be testable code.



## Problem

A polished answer can still bypass approval or contain invalid citations.


## Concept

Objective checks encode non-negotiable behavior before subjective scoring.

An evaluation case is a named input and a measurable check. Separate objective safety contracts from subjective answer quality, because prose can look good while a boundary is broken.


## Abstraction

test cases → deterministic checks → pass/fail evidence. This is the smallest new layer introduced by this lesson; the earlier layers remain in place.


## Architecture

```text
known cases ──▶ deterministic harness checks
                   ├──▶ ROI contract
                   ├──▶ report/source contract
                   └──▶ approval boundary
                              │
                         PASS / FAIL
```

Follow the arrows while executing the cells. A tool result or validation failure is an observation, not an instruction that the model may bypass.


## Step 1 — Define independent cases and their plain-Python pass/fail conditions

Define independent cases and their plain-Python pass/fail conditions.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[2] if "__file__" in globals() else Path.cwd()))

from workshop.control import mock_email, validate_report
from workshop.schemas import DecisionReport, Evidence
from workshop.tools import calculate_annual_roi

cases = [
    ("roi scenario", lambda: calculate_annual_roi(45_000, 30_000) == 50.0),
    ("zero cost rejected", lambda: rejects_zero_cost()),
    ("email blocked", lambda: mock_email(example_report, approved=False, action_id="eval-denied").startswith("BLOCKED")),
    ("citations valid", lambda: validate_report(example_report)[0]),
]


def rejects_zero_cost() -> bool:
    try:
        calculate_annual_roi(45_000, 0)
    except ValueError:
        return True
    return False


example_report = DecisionReport(
    recommendation="pilot",
    rationale="The possible benefit merits a pilot, while security gaps block full adoption.",
    evidence=[
        Evidence(source_id="pricing.md", claim="Annual benefit is estimated at $45,000."),
        Evidence(source_id="security.md", claim="Deletion guarantees are unverified."),
    ],
)



## Run the checks
Extend the set with a model run after establishing this deterministic base.



## Step 2 — Run every check and use a failing case to identify the broken boundary

Run every check and use a failing case to identify the broken boundary.


In [ ]:
for name, check in cases:
    print(("PASS" if check() else "FAIL"), name)


## Execution flow

test cases → deterministic checks → pass/fail evidence. Read the printed result together with the task, selected tool, state, or trace. If the result differs, locate the first boundary where the observed value changed.


## Failure mode to know about

A prose-quality score cannot catch a bypassed approval gate. Keep deterministic safety checks as release blockers and use model-judged scores only as an additional signal.


## Production implication

Run safety and contract cases in CI. Keep a small labeled set of grounded-answer examples for model or prompt changes.


## Your turn

Add a case for a report containing an unknown source ID.

**Checkpoint:** A failing case identifies a specific harness boundary to repair.
